In [ ]:
import psutil
import torch
from ultralytics import YOLO, utils

utils.checks.collect_system_info()

In [ ]:
model = YOLO("yolo26n.pt")


def get_comp_state():
    mps_alloc = torch.mps.current_allocated_memory() / 1e9
    mps_driver = torch.mps.driver_allocated_memory() / 1e9
    ram = psutil.virtual_memory()
    ram_used = (ram.total - ram.available) / 1e9
    ram_total = ram.total / 1e9
    swap = psutil.swap_memory()
    swap_used = swap.used / 1e9
    return f"MPS alloc= {mps_alloc:.2f} GB | MPS driver= {mps_driver:.2f} GB | RAM= {ram_used:.1f}/{ram_total:.1f} GB ({ram.percent:.1f}%) | swap= {swap_used:.2f} GB"


def on_train_batch_end(_):
    # global _batch_counter
    # _batch_counter += 1
    # if _batch_counter % 250 == 0:
    print(f"   {get_comp_state()}", flush=True)


def on_train_epoch_end(trainer):
    print(f"Epoch {trainer.epoch+1} END:   {get_comp_state()}", flush=True)


def on_train_epoch_start(trainer):
    # global _batch_counter
    # _batch_counter = 0
    print(f"Epoch {trainer.epoch+1} START: {get_comp_state()}", flush=True)


model.add_callback("on_train_batch_end", on_train_batch_end)
model.add_callback("on_train_epoch_end", on_train_epoch_end)
model.add_callback("on_train_epoch_start", on_train_epoch_start)

model.train(
    data="coco4000.yaml",
    epochs=3,
    batch=64,
    device="mps",
    workers=0,
    freeze=23,
)